In [0]:
dbutils.widgets.text("landing_base", "/Volumes/workspace/mf_insight/mf_insight_raw", "Raw landing zone path")
dbutils.widgets.text("database_name", "mf_insight", "Database (schema) name")
dbutils.widgets.text("funds_per_category", "4", "Funds to pick per category")

LANDING_BASE = dbutils.widgets.get("landing_base")
DATABASE_NAME = dbutils.widgets.get("database_name")
FUNDS_PER_CATEGORY = int(dbutils.widgets.get("funds_per_category"))

TARGET_CATEGORIES = {
    "large_cap": "Equity Scheme - Large Cap Fund",
    "mid_cap":   "Equity Scheme - Mid Cap Fund",
    "debt":      "Debt Scheme - Short Duration Fund",
    "hybrid":    "Hybrid Scheme - Aggressive Hybrid Fund",
}

AMFI_NAV_ALL_URL = "https://www.amfiindia.com/spages/NAVAll.txt"
MFAPI_BASE = "https://api.mfapi.in/mf"

print(f"Landing zone : {LANDING_BASE}")
print(f"Database     : {DATABASE_NAME}")
print(f"Per category : {FUNDS_PER_CATEGORY}")

Landing zone : /Volumes/workspace/mf_insight/mf_insight_raw
Database     : mf_insight
Per category : 4


In [0]:
import requests
import re

def fetch_amfi_scheme_master():
    resp = requests.get(AMFI_NAV_ALL_URL, timeout=60)
    resp.raise_for_status()
    lines = resp.text.splitlines()

    rows = []
    current_category = None
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if ";" not in line:
            if "Scheme" in line:
                current_category = line
            continue

        parts = line.split(";")
        if len(parts) < 6:
            continue

        scheme_code, isin_div_payout, isin_div_reinvest, scheme_name, nav, nav_date = parts[:6]
        if not scheme_code.strip().isdigit():
            continue

        rows.append({
            "scheme_code": scheme_code.strip(),
            "scheme_name": scheme_name.strip(),
            "category_raw": current_category or "Unknown",
            "amc_name": scheme_name.split(" ")[0].strip(),
        })
    return rows

amfi_rows = fetch_amfi_scheme_master()
print(f"Parsed {len(amfi_rows)} scheme rows from AMFI master.")
print("Sample categories seen:")
for c in sorted(set(r["category_raw"] for r in amfi_rows))[:10]:
    print(" -", c)

Parsed 14224 scheme rows from AMFI master.
Sample categories seen:
 - Close Ended Schemes(ELSS)
 - Close Ended Schemes(Growth)
 - Close Ended Schemes(Income)
 - Interval Fund Schemes(Income)
 - Open Ended Schemes(Debt Scheme - Banking and PSU Fund)
 - Open Ended Schemes(Debt Scheme - Corporate Bond Fund)
 - Open Ended Schemes(Debt Scheme - Credit Risk Fund)
 - Open Ended Schemes(Debt Scheme - Dynamic Bond)
 - Open Ended Schemes(Debt Scheme - Floater Fund)
 - Open Ended Schemes(Debt Scheme - Gilt Fund with 10 year constant duration)


In [0]:
selected_funds = []

for label, category_substr in TARGET_CATEGORIES.items():
    matches = [r for r in amfi_rows if category_substr.lower() in r["category_raw"].lower()]
    growth_matches = [m for m in matches if "growth" in m["scheme_name"].lower()]
    pool = growth_matches if growth_matches else matches

    chosen = pool[:FUNDS_PER_CATEGORY]
    for c in chosen:
        c["category"] = label
    selected_funds.extend(chosen)

    print(f"{label:10s} -> matched {len(matches)} schemes, selected {len(chosen)}")

print(f"\nTotal funds selected for demo dataset: {len(selected_funds)}")
for f in selected_funds:
    print(f"  [{f['category']:10s}] {f['scheme_code']:>8s}  {f['scheme_name']}")

large_cap  -> matched 163 schemes, selected 4
mid_cap    -> matched 131 schemes, selected 4
debt       -> matched 250 schemes, selected 4
hybrid     -> matched 189 schemes, selected 4

Total funds selected for demo dataset: 16
  [large_cap ]   119528  Aditya Birla Sun Life Large Cap Fund - Growth - Direct Plan
  [large_cap ]   103174  Aditya Birla Sun Life Large Cap Fund-Growth
  [large_cap ]   120465  Axis Large Cap Fund - Direct Plan - Growth
  [large_cap ]   112277  Axis Large Cap Fund - Regular Plan - Growth
  [mid_cap   ]   119620  Aditya Birla Sun Life Midcap Fund - Growth - Direct Plan
  [mid_cap   ]   101592  Aditya Birla Sun Life MIDCAP Fund-Growth
  [mid_cap   ]   120505  Axis Midcap Fund - Direct Plan - Growth
  [mid_cap   ]   114564  Axis Midcap Fund - Regular Plan - Growth
  [debt      ]   119498  Aditya Birla Sun Life Short Term Fund - Growth - Direct Plan
  [debt      ]   101844  Aditya Birla Sun Life Short Term Fund - Growth - Regular Plan
  [debt      ]   120510  Axis 

In [0]:
import json
import time
import os

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.mf_insight.mf_insight_raw")

def fetch_nav_history(scheme_code: str) -> dict:
    url = f"{MFAPI_BASE}/{scheme_code}"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return resp.json()

landed_files = []
fetch_errors = []

for f in selected_funds:
    code = f["scheme_code"]
    try:
        payload = fetch_nav_history(code)
        out_path = os.path.join(LANDING_BASE, f"{code}.json")
        with open(out_path, "w") as fh:
            json.dump(payload, fh)
        n_points = len(payload.get("data", []))
        landed_files.append({"scheme_code": code, "path": out_path, "nav_points": n_points})
        print(f"OK   {code:>8s}  {n_points:5d} NAV points  -> {out_path}")
    except Exception as e:
        fetch_errors.append({"scheme_code": code, "error": str(e)})
        print(f"FAIL {code:>8s}  {e}")
    time.sleep(0.3)

print(f"\nLanded {len(landed_files)} files, {len(fetch_errors)} errors.")
if fetch_errors:
    print("Errors:", fetch_errors)

OK     119528   3313 NAV points  -> /Volumes/workspace/mf_insight/mf_insight_raw/119528.json
OK     103174   4977 NAV points  -> /Volumes/workspace/mf_insight/mf_insight_raw/103174.json
OK     120465   3323 NAV points  -> /Volumes/workspace/mf_insight/mf_insight_raw/120465.json
OK     112277   4063 NAV points  -> /Volumes/workspace/mf_insight/mf_insight_raw/112277.json
OK     119620   3313 NAV points  -> /Volumes/workspace/mf_insight/mf_insight_raw/119620.json
OK     101592   4977 NAV points  -> /Volumes/workspace/mf_insight/mf_insight_raw/101592.json
OK     120505   3323 NAV points  -> /Volumes/workspace/mf_insight/mf_insight_raw/120505.json
OK     114564   3779 NAV points  -> /Volumes/workspace/mf_insight/mf_insight_raw/114564.json
OK     119498   3252 NAV points  -> /Volumes/workspace/mf_insight/mf_insight_raw/119498.json
OK     101844   4892 NAV points  -> /Volumes/workspace/mf_insight/mf_insight_raw/101844.json
OK     120510   3258 NAV points  -> /Volumes/workspace/mf_insight/mf_i

In [0]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}")

fund_master_df = spark.createDataFrame(selected_funds).select(
    "scheme_code", "scheme_name", "category", "amc_name"
)

(fund_master_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{DATABASE_NAME}.fund_master"))

print(f"FUND_MASTER written: {fund_master_df.count()} rows")
display(fund_master_df)

FUND_MASTER written: 16 rows


scheme_code,scheme_name,category,amc_name
119528,Aditya Birla Sun Life Large Cap Fund - Growth - Direct Plan,large_cap,Aditya
103174,Aditya Birla Sun Life Large Cap Fund-Growth,large_cap,Aditya
120465,Axis Large Cap Fund - Direct Plan - Growth,large_cap,Axis
112277,Axis Large Cap Fund - Regular Plan - Growth,large_cap,Axis
119620,Aditya Birla Sun Life Midcap Fund - Growth - Direct Plan,mid_cap,Aditya
101592,Aditya Birla Sun Life MIDCAP Fund-Growth,mid_cap,Aditya
120505,Axis Midcap Fund - Direct Plan - Growth,mid_cap,Axis
114564,Axis Midcap Fund - Regular Plan - Growth,mid_cap,Axis
119498,Aditya Birla Sun Life Short Term Fund - Growth - Direct Plan,debt,Aditya
101844,Aditya Birla Sun Life Short Term Fund - Growth - Regular Plan,debt,Aditya


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

raw_nav_df = spark.read.option("multiLine", True).json(LANDING_BASE + "/*.json")

nav_history_df = (
    raw_nav_df
    .select(
        F.col("meta.scheme_code").alias("scheme_code"),
        F.col("meta.scheme_name").alias("scheme_name"),
        F.explode("data").alias("nav_point"),
    )
    .select(
        "scheme_code",
        "scheme_name",
        F.to_date(F.col("nav_point.date"), "dd-MM-yyyy").alias("nav_date"),
        F.col("nav_point.nav").cast("double").alias("nav"),
    )
    .filter(F.col("nav_date").isNotNull())
)

(nav_history_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("scheme_code")
    .saveAsTable(f"{DATABASE_NAME}.nav_history"))

print(f"NAV_HISTORY written: {nav_history_df.count()} rows across {nav_history_df.select('scheme_code').distinct().count()} funds")
display(nav_history_df.limit(10))

NAV_HISTORY written: 58606 rows across 16 funds


scheme_code,scheme_name,nav_date,nav
103155,Aditya Birla Sun Life Equity Hybrid'95 Fund - Regular Plan-Growth,2026-06-23,1509.44
103155,Aditya Birla Sun Life Equity Hybrid'95 Fund - Regular Plan-Growth,2026-06-22,1517.0
103155,Aditya Birla Sun Life Equity Hybrid'95 Fund - Regular Plan-Growth,2026-06-19,1510.63
103155,Aditya Birla Sun Life Equity Hybrid'95 Fund - Regular Plan-Growth,2026-06-18,1512.83
103155,Aditya Birla Sun Life Equity Hybrid'95 Fund - Regular Plan-Growth,2026-06-17,1507.61
103155,Aditya Birla Sun Life Equity Hybrid'95 Fund - Regular Plan-Growth,2026-06-16,1503.24
103155,Aditya Birla Sun Life Equity Hybrid'95 Fund - Regular Plan-Growth,2026-06-15,1497.21
103155,Aditya Birla Sun Life Equity Hybrid'95 Fund - Regular Plan-Growth,2026-06-12,1481.66
103155,Aditya Birla Sun Life Equity Hybrid'95 Fund - Regular Plan-Growth,2026-06-11,1459.26
103155,Aditya Birla Sun Life Equity Hybrid'95 Fund - Regular Plan-Growth,2026-06-10,1463.67


In [0]:
summary_df = (
    nav_history_df
    .groupBy("scheme_code", "scheme_name")
    .agg(
        F.min("nav_date").alias("earliest_date"),
        F.max("nav_date").alias("latest_date"),
        F.count("*").alias("num_points"),
        F.sum(F.when(F.col("nav").isNull(), 1).otherwise(0)).alias("null_navs"),
    )
    .orderBy("scheme_code")
)

display(summary_df)

null_count = summary_df.agg(F.sum("null_navs")).first()[0]
print(f"Total null NAV values across all funds: {null_count}")
print("Note: gaps on weekends/holidays are expected (NAV not published) — not a bug.")
print("Duplicate-date checks and AMC cross-matching are flagged for Day 2 hardening.")

scheme_code,scheme_name,earliest_date,latest_date,num_points,null_navs
101592,Aditya Birla Sun Life MIDCAP Fund-Growth,2006-04-03,2026-06-23,4977,0
101844,Aditya Birla Sun Life Short Term Fund - Growth - Regular Plan,2006-04-03,2026-06-23,4892,0
103155,Aditya Birla Sun Life Equity Hybrid'95 Fund - Regular Plan-Growth,2006-04-03,2026-06-23,4977,0
103174,Aditya Birla Sun Life Large Cap Fund-Growth,2006-04-03,2026-06-23,4977,0
112277,Axis Large Cap Fund - Regular Plan - Growth,2010-01-07,2026-06-23,4063,0
112354,Axis Short Duration Fund - Regular Plan - Growth Option,2010-01-25,2026-06-23,3969,0
114564,Axis Midcap Fund - Regular Plan - Growth,2011-02-24,2026-06-23,3779,0
119498,Aditya Birla Sun Life Short Term Fund - Growth - Direct Plan,2013-01-02,2026-06-23,3252,0
119528,Aditya Birla Sun Life Large Cap Fund - Growth - Direct Plan,2013-01-02,2026-06-23,3313,0
119620,Aditya Birla Sun Life Midcap Fund - Growth - Direct Plan,2013-01-02,2026-06-23,3313,0


Total null NAV values across all funds: 0
Note: gaps on weekends/holidays are expected (NAV not published) — not a bug.
Duplicate-date checks and AMC cross-matching are flagged for Day 2 hardening.
